# Interactive fisheye rectification tuner

This notebook compares three inverse-mapping methods for an uncalibrated fisheye image:

1. **Three perspective rectifications** with independently aimed virtual cameras, cropped and stitched into one image.
2. **Cylindrical projection** using rays `[sin(azimuth), Y, cos(azimuth)]`.
3. **Cylindrical projection with extrinsic rotation** (yaw, pitch, and roll).

The geometry follows the camera convention **X right, Y down, Z forward**. All sliders use inverse mapping, so every output pixel asks where to sample the source fisheye image. The source intrinsics are approximated by the circle/ellipse center, X/Y radii, lens model, source FOV, and optional radial correction coefficients.

> Manual tuning can produce a visually useful rectification, but it is not a metric camera calibration. Do not use the tuned mapping for precise 3D projection until it has been validated against calibration targets or known geometry.

Reference: [Fisheye camera model tutorial - multiple rectifications and cylindrical images](https://plaut.github.io/fisheye_tutorial/).

## Dependencies

Run the notebook from the repository environment. If widgets are missing, install them with:

```bash
pip install opencv-python ipywidgets matplotlib
```

In classic Jupyter Notebook, an older installation may also require `jupyter nbextension enable --py widgetsnbextension`. JupyterLab 3+ normally needs no extra activation.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.patches import Ellipse
from PIL import Image


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'autolabeler').is_dir():
            return candidate
    raise FileNotFoundError('Could not find repository root containing src/autolabeler')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from autolabeler.camera.fisheye_tuning import (
    build_tunable_fisheye_remap,
    crop_and_stitch_three_views,
    remap_image,
)

plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 11})
print('REPO_ROOT:', REPO_ROOT)

## Load one fisheye image

Set `IMAGE_PATH` and rerun this cell. OpenCV decoding is used so non-ASCII filesystem paths also work.

In [ ]:
IMAGE_PATH = Path('/path/to/fisheye.jpg')


def read_rgb(path: Path) -> np.ndarray:
    path = Path(path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f'Image not found: {path}')
    encoded = np.fromfile(path, dtype=np.uint8)
    bgr = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f'OpenCV could not decode: {path}')
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


SOURCE_RGB = read_rgb(IMAGE_PATH)
SOURCE_HEIGHT, SOURCE_WIDTH = SOURCE_RGB.shape[:2]
print(f'Loaded {IMAGE_PATH}: {SOURCE_WIDTH}x{SOURCE_HEIGHT}')

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(SOURCE_RGB)
ax.set_title('Source fisheye image')
ax.axis('off')
plt.show()

## Interactive controls

Useful tuning order:

1. Fit `source cx`, `source cy`, `radius X`, and `radius Y` to the valid fisheye circle.
2. Select the lens model and tune `source FOV`. Start with `equalarea` for an equisolid lens or `linear` for an equidistant lens.
3. Tune output horizontal/vertical FOV and principal point.
4. Use radial `k1..k4` only after the basic circle and FOV are close.
5. In extrinsic mode, tune yaw, pitch, and roll so the horizon is horizontal and vertical structures are upright.

`continuous_update=False` is used, so expensive remapping happens when a slider is released.

In [ ]:
slider_layout = widgets.Layout(width='420px')
description_style = {'description_width': '175px'}


def float_slider(description, value, minimum, maximum, step):
    return widgets.FloatSlider(
        description=description, value=value, min=minimum, max=maximum, step=step,
        continuous_update=False, readout_format='.3f', layout=slider_layout, style=description_style,
    )


def int_slider(description, value, minimum, maximum, step=1):
    return widgets.IntSlider(
        description=description, value=value, min=minimum, max=maximum, step=step,
        continuous_update=False, layout=slider_layout, style=description_style,
    )


method = widgets.ToggleButtons(
    options=[
        ('Three perspective views', 'three'),
        ('Cylindrical', 'cylindrical'),
        ('Cylindrical + extrinsic', 'cylindrical_extrinsic'),
    ],
    value='three', description='Method', style={'description_width': '70px'},
)
fisheye_model = widgets.Dropdown(
    options=['linear', 'equalarea', 'orthographic', 'stereographic'],
    value='equalarea', description='fisheye model', layout=slider_layout, style=description_style,
)
interpolation = widgets.Dropdown(
    options=['nearest', 'linear', 'cubic', 'lanczos'],
    value='lanczos', description='interpolation', layout=slider_layout, style=description_style,
)
source_cx = float_slider('source cx [px]', SOURCE_WIDTH / 2.0, 0.0, SOURCE_WIDTH - 1.0, 1.0)
source_cy = float_slider('source cy [px]', SOURCE_HEIGHT / 2.0, 0.0, SOURCE_HEIGHT - 1.0, 1.0)
default_radius = min(SOURCE_WIDTH, SOURCE_HEIGHT) * 0.48
source_radius_x = float_slider('source radius X [px]', default_radius, 10.0, SOURCE_WIDTH, 1.0)
source_radius_y = float_slider('source radius Y [px]', default_radius, 10.0, SOURCE_HEIGHT, 1.0)
source_fov = float_slider('source full FOV [deg]', 190.0, 90.0, 260.0, 1.0)
k1 = float_slider('radial k1', 0.0, -0.50, 0.50, 0.005)
k2 = float_slider('radial k2', 0.0, -0.30, 0.30, 0.005)
k3 = float_slider('radial k3', 0.0, -0.20, 0.20, 0.005)
k4 = float_slider('radial k4', 0.0, -0.10, 0.10, 0.002)

output_width = int_slider('output width [px]', min(1920, SOURCE_WIDTH), 320, max(4096, SOURCE_WIDTH), 16)
output_height = int_slider('output height [px]', min(1080, SOURCE_HEIGHT), 240, max(2160, SOURCE_HEIGHT), 16)
output_hfov = float_slider('output horizontal FOV', 180.0, 30.0, 250.0, 1.0)
output_vfov = float_slider('output vertical FOV', 105.0, 20.0, 170.0, 1.0)
output_cx = float_slider('output cx / width', 0.5, 0.0, 1.0, 0.005)
output_cy = float_slider('output cy / height', 0.5, 0.0, 1.0, 0.005)

extrinsic_yaw = float_slider('extrinsic yaw [deg]', 0.0, -180.0, 180.0, 0.5)
extrinsic_pitch = float_slider('extrinsic pitch [deg]', 0.0, -90.0, 90.0, 0.5)
extrinsic_roll = float_slider('extrinsic roll [deg]', 0.0, -90.0, 90.0, 0.5)

panel_width = int_slider('panel width [px]', min(960, SOURCE_WIDTH), 320, max(1920, SOURCE_WIDTH), 16)
panel_height = int_slider('panel height [px]', min(720, SOURCE_HEIGHT), 240, max(1536, SOURCE_HEIGHT), 16)
panel_hfov = float_slider('panel horizontal FOV', 90.0, 30.0, 170.0, 1.0)
panel_vfov = float_slider('panel vertical FOV', 85.0, 20.0, 170.0, 1.0)
three_center_yaw = float_slider('center-view yaw [deg]', 0.0, -90.0, 90.0, 0.5)
three_yaw_separation = float_slider('side yaw separation', 45.0, 5.0, 100.0, 1.0)
three_pitch = float_slider('view pitch [deg]', 0.0, -75.0, 75.0, 0.5)
three_roll = float_slider('view roll [deg]', 0.0, -45.0, 45.0, 0.5)
overlap_fraction = float_slider('crop overlap fraction', 0.10, 0.0, 0.60, 0.01)

COMMON_WIDGETS = [
    fisheye_model, interpolation, source_cx, source_cy, source_radius_x, source_radius_y,
    source_fov, k1, k2, k3, k4,
]
CYLINDER_WIDGETS = [output_width, output_height, output_hfov, output_vfov, output_cx, output_cy]
EXTRINSIC_WIDGETS = [extrinsic_yaw, extrinsic_pitch, extrinsic_roll]
THREE_WIDGETS = [
    panel_width, panel_height, panel_hfov, panel_vfov, three_center_yaw,
    three_yaw_separation, three_pitch, three_roll, overlap_fraction,
]
specific_controls = widgets.VBox()


def update_specific_controls(*_):
    if method.value == 'three':
        specific_controls.children = tuple(THREE_WIDGETS)
    elif method.value == 'cylindrical':
        specific_controls.children = tuple(CYLINDER_WIDGETS)
    else:
        specific_controls.children = tuple(CYLINDER_WIDGETS + EXTRINSIC_WIDGETS)


method.observe(update_specific_controls, names='value')
update_specific_controls()
display(method)
display(widgets.HBox([
    widgets.VBox([widgets.HTML('<b>Source fisheye model</b>'), *COMMON_WIDGETS]),
    widgets.VBox([widgets.HTML('<b>Selected output model</b>'), specific_controls]),
]))

In [ ]:
LAST_RESULT: np.ndarray | None = None
LAST_PARAMETERS: dict = {}


def source_parameters(p: dict) -> dict:
    return {
        'input_shape': SOURCE_RGB.shape[:2],
        'source_cx': p['source_cx'],
        'source_cy': p['source_cy'],
        'source_radius_x': p['source_radius_x'],
        'source_radius_y': p['source_radius_y'],
        'source_fov': p['source_fov'],
        'fisheye_model': p['fisheye_model'],
        'k1': p['k1'], 'k2': p['k2'], 'k3': p['k3'], 'k4': p['k4'],
    }


def draw_source(ax, p: dict) -> None:
    ax.imshow(SOURCE_RGB)
    ellipse = Ellipse(
        (p['source_cx'], p['source_cy']),
        2.0 * p['source_radius_x'], 2.0 * p['source_radius_y'],
        fill=False, edgecolor='lime', linewidth=1.5,
    )
    ax.add_patch(ellipse)
    ax.scatter([p['source_cx']], [p['source_cy']], c='red', s=18)
    ax.set_title('Source circle / ellipse')
    ax.axis('off')


def render_current(**p):
    global LAST_RESULT, LAST_PARAMETERS
    common = source_parameters(p)
    try:
        if p['method'] == 'three':
            yaws = [
                p['three_center_yaw'] - p['three_yaw_separation'],
                p['three_center_yaw'],
                p['three_center_yaw'] + p['three_yaw_separation'],
            ]
            remaps = [
                build_tunable_fisheye_remap(
                    **common, output_shape=(p['panel_height'], p['panel_width']),
                    projection='perspective', horizontal_fov=p['panel_hfov'],
                    vertical_fov=p['panel_vfov'], yaw=yaw, pitch=p['three_pitch'],
                    roll=p['three_roll'],
                )
                for yaw in yaws
            ]
            views = tuple(remap_image(SOURCE_RGB, item, interpolation=p['interpolation']) for item in remaps)
            overlap_pixels = min(
                p['panel_width'] - 1, round(p['panel_width'] * p['overlap_fraction'])
            )
            result = crop_and_stitch_three_views(views, overlap_pixels=overlap_pixels)
            fig = plt.figure(figsize=(18, 10), constrained_layout=True)
            grid = fig.add_gridspec(2, 3)
            draw_source(fig.add_subplot(grid[0, 0]), p)
            stitched_ax = fig.add_subplot(grid[0, 1:])
            stitched_ax.imshow(result)
            stitched_ax.set_title(
                f'Three-view stitched output ({result.shape[1]}x{result.shape[0]}, crop={overlap_pixels}px)'
            )
            stitched_ax.axis('off')
            for index, (view, yaw, remap) in enumerate(zip(views, yaws, remaps)):
                ax = fig.add_subplot(grid[1, index])
                ax.imshow(view)
                valid_percent = 100.0 * float(remap.valid.mean())
                ax.set_title(f'view {index + 1}: yaw={yaw:.1f} deg, valid={valid_percent:.1f}%')
                ax.axis('off')
            parameters = {
                **p, 'projection': 'three_perspective', 'view_yaws': yaws,
                'overlap_pixels': overlap_pixels, 'result_shape': list(result.shape),
                'valid_fraction_per_view': [float(item.valid.mean()) for item in remaps],
            }
        else:
            use_extrinsic = p['method'] == 'cylindrical_extrinsic'
            yaw = p['extrinsic_yaw'] if use_extrinsic else 0.0
            pitch = p['extrinsic_pitch'] if use_extrinsic else 0.0
            roll = p['extrinsic_roll'] if use_extrinsic else 0.0
            remap = build_tunable_fisheye_remap(
                **common, output_shape=(p['output_height'], p['output_width']),
                projection='cylindrical', horizontal_fov=p['output_hfov'],
                vertical_fov=p['output_vfov'], output_cx=p['output_cx'],
                output_cy=p['output_cy'], yaw=yaw, pitch=pitch, roll=roll,
            )
            result = remap_image(SOURCE_RGB, remap, interpolation=p['interpolation'])
            fig, axes = plt.subplots(1, 2, figsize=(18, 8), constrained_layout=True)
            draw_source(axes[0], p)
            axes[1].imshow(result)
            valid_percent = 100.0 * float(remap.valid.mean())
            suffix = ' + extrinsic rotation' if use_extrinsic else ''
            axes[1].set_title(
                f'Cylindrical{suffix}: {result.shape[1]}x{result.shape[0]}, valid={valid_percent:.1f}%'
            )
            axes[1].axis('off')
            parameters = {
                **p, 'projection': 'cylindrical', 'applied_yaw': yaw,
                'applied_pitch': pitch, 'applied_roll': roll,
                'result_shape': list(result.shape), 'valid_fraction': float(remap.valid.mean()),
            }
        LAST_RESULT = result
        LAST_PARAMETERS = parameters
        plt.show()
    except Exception as error:
        LAST_RESULT = None
        LAST_PARAMETERS = {}
        print(f'Rectification failed: {type(error).__name__}: {error}')


CONTROLS = {
    'method': method, 'fisheye_model': fisheye_model, 'interpolation': interpolation,
    'source_cx': source_cx, 'source_cy': source_cy,
    'source_radius_x': source_radius_x, 'source_radius_y': source_radius_y,
    'source_fov': source_fov, 'k1': k1, 'k2': k2, 'k3': k3, 'k4': k4,
    'output_width': output_width, 'output_height': output_height,
    'output_hfov': output_hfov, 'output_vfov': output_vfov,
    'output_cx': output_cx, 'output_cy': output_cy,
    'extrinsic_yaw': extrinsic_yaw, 'extrinsic_pitch': extrinsic_pitch,
    'extrinsic_roll': extrinsic_roll, 'panel_width': panel_width,
    'panel_height': panel_height, 'panel_hfov': panel_hfov, 'panel_vfov': panel_vfov,
    'three_center_yaw': three_center_yaw, 'three_yaw_separation': three_yaw_separation,
    'three_pitch': three_pitch, 'three_roll': three_roll,
    'overlap_fraction': overlap_fraction,
}
rectification_output = widgets.interactive_output(render_current, CONTROLS)
display(rectification_output)

## Save the current rectification and parameters

The JSON sidecar records every slider value, the selected method, output shape, and valid-pixel fraction.

In [ ]:
save_path = widgets.Text(
    value=str(REPO_ROOT / 'output' / 'fisheye_tuning' / 'rectified.png'),
    description='output image', layout=widgets.Layout(width='720px'),
    style={'description_width': '100px'},
)
save_button = widgets.Button(description='Save current result', button_style='primary')
save_status = widgets.Output()


def save_current(_):
    with save_status:
        save_status.clear_output()
        if LAST_RESULT is None:
            print('No valid result is available. Adjust the controls and render again.')
            return
        destination = Path(save_path.value).expanduser().resolve()
        destination.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(np.asarray(LAST_RESULT, dtype=np.uint8)).save(destination)
        parameter_path = destination.with_suffix(destination.suffix + '.json')
        payload = {
            'source_image': str(Path(IMAGE_PATH).expanduser().resolve()),
            'source_shape': list(SOURCE_RGB.shape),
            'parameters': LAST_PARAMETERS,
        }
        parameter_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
        print('Saved image:', destination)
        print('Saved parameters:', parameter_path)


save_button.on_click(save_current)
display(widgets.HBox([save_path, save_button]), save_status)

## What to inspect

- **Straightness:** poles and building edges should be straight in each perspective panel; vertical lines should remain straight in cylindrical output.
- **Scale continuity:** in the three-view method, adjust yaw separation, panel FOV, and crop overlap so objects do not abruptly duplicate or disappear at seams. A model discontinuity remains at every seam by construction.
- **Horizon:** use extrinsic pitch/roll and output `cy` to make the horizon level without wasting most of the output on empty pixels.
- **Coverage:** black output regions mean the requested virtual ray is outside the fitted source lens/FOV. Reduce output FOV, correct source radius/FOV, or move the output principal point.
- **Radial coefficients:** large `k` values can overfit one part of the image. Prefer the simplest lens model and keep coefficients near zero unless line curvature clearly improves across the whole frame.